In [137]:
#Library
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import statsmodels.api as sm
from pmdarima import auto_arima
import matplotlib.pyplot as plt

In [138]:
# Load Dataset
df = pd.read_excel("dataset baru.xlsx")

# Pastikan format tanggal benar
df["TGL/BLN/THN"] = pd.to_datetime(df["TGL/BLN/THN"], dayfirst=True, errors="coerce")

# Hapus baris tanggal rusak & sort
df = df.dropna(subset=["TGL/BLN/THN"])
df = df.sort_values("TGL/BLN/THN")

# ---------------------------
# 🔧 BERSIHKAN kolom NYATA
# ---------------------------

# Ubah ke string dulu
df["NYATA"] = df["NYATA"].astype(str).str.upper().str.strip()

# Ganti 'LIBUR' menjadi 0
df["NYATA"] = df["NYATA"].replace("LIBUR", "0")

# Hilangkan karakter asing (misal koma/dot format lokal)
df["NYATA"] = df["NYATA"].str.replace(",", ".", regex=False)
df["NYATA"] = df["NYATA"].str.replace("[^0-9.]", "", regex=True)

# Convert ke float
df["NYATA"] = pd.to_numeric(df["NYATA"], errors="coerce")

# Hapus baris kosong setelah dibersihkan
df = df.dropna(subset=["NYATA"])

# 🔧 AMBIL DATA MODEL s/d 27-11-2025
df_model = df[df["TGL/BLN/THN"] <= "2025-11-27"]

df_model = df_model.set_index("TGL/BLN/THN")

# Kolom time series
y = df_model["NYATA"].astype(float)

df_model.head()


,NYATA
TGL/BLN/THN,
2025-01-01,2
2025-01-02,2
2025-01-03,3
2025-01-04,3
2025-01-05,3


In [139]:
# FITTING MODEL ARIMA
model = auto_arima(y, seasonal=False, trace=False)
model.fit(y)

ARIMA(order=(2, 1, 1), scoring_args={}, suppress_warnings=True,
      with_intercept=False)

In [140]:
# PREDIKSI N HARI KE DEPAN
N = 30  # jumlah hari prediksi, bisa diubah

# Buat tanggal prediksi (mulai 28-11-2025)
start_date = pd.to_datetime("2025-11-28")
future_dates = pd.date_range(start=start_date, periods=N, freq="D")

# Prediksi nilai NYATA
forecast = model.predict(n_periods=N)

df_forecast = pd.DataFrame({
    "Tanggal": future_dates,
    "Prediksi_NYATA": forecast
}).set_index("Tanggal")

In [141]:
# BULATKAN KE ATAS UNTUK JUMLAH ADONAN
ADONAN_PER_PCS = 1
df_forecast["Adonan"] = np.ceil(df_forecast["Prediksi_NYATA"] / ADONAN_PER_PCS)

In [142]:
# ARRAY BAHAN, STOK, SATUAN, KEBUTUHAN PER ADONAN, MINIMAL STOK
nama_bahan = np.array(["Tepung", "MargarinA", "MargarinB", "Telur", "Air", "Gula", "Fermipan"])
satuan = np.array(["kg", "gram", "gram", "pcs", "ml", "gram", "sdm"])

# Stok awal masing-masing bahan (contoh)
stok_awal = np.array([50, 2000, 2500, 80, 16000, 4000, 30])

# Kebutuhan per 1 adonan
kebutuhan_per_adonan = np.array([1, 100, 100, 8, 800, 400, 1])

# Minimal stok yang diperbolehkan
stok_minimum = np.array([5, 500, 500, 40, 4000, 2000, 5])

In [143]:
# HITUNG KEBUTUHAN BAHAN PER TANGGAL PREDIKSI
for i, bahan in enumerate(nama_bahan):
    df_forecast[f"Kebutuhan_{bahan}"] = df_forecast["Adonan"] * kebutuhan_per_adonan[i]

In [144]:
# HITUNG SISA STOK SETIAP HARI
stok_sisa = pd.DataFrame(index=df_forecast.index, columns=nama_bahan)

stok_current = stok_awal.copy()

for date in df_forecast.index:
    # Hitung konsumsi bahan pada hari itu
    konsumsi = df_forecast.loc[date, [f"Kebutuhan_{bahan}" for bahan in nama_bahan]].values
    
    # Update stok
    stok_current = stok_current - konsumsi
    stok_sisa.loc[date] = stok_current

# Tambahkan ke df_forecast
for i, bahan in enumerate(nama_bahan):
    df_forecast[f"StokSisa_{bahan}"] = stok_sisa[bahan]

In [145]:
# CARI TANGGAL DIMANA STOK MENYENTUH BATAS MINIMUM
batas_tanggal = {}

for i, bahan in enumerate(nama_bahan):
    sisa = stok_sisa[bahan].astype(float)
    min_stock = stok_minimum[i]

    mask = sisa <= min_stock
    if mask.any():
        tanggal_krisis = sisa[mask].index[0]
        batas_tanggal[bahan] = tanggal_krisis
    else:
        batas_tanggal[bahan] = "AMAN (tidak turun di bawah min)"

In [146]:
# CARI NAMA KOLOM PEMBULATAN
if "Pembulatan" in df_forecast.columns:
    kolom_bulat = "Pembulatan"
elif "Bulat" in df_forecast.columns:
    kolom_bulat = "Bulat"
elif "Adonan" in df_forecast.columns:
    kolom_bulat = "Adonan"
else:
    kolom_bulat = None
    print("⚠ Tidak ditemukan kolom pembulatan!")

print("=== HASIL PREDIKSI DENGAN BAHAN & STOK ===\n")

for idx, row in df_forecast.iterrows():

    print(f"Tanggal : {idx.date()}")
    print(f"Prediksi : {row['Prediksi_NYATA']:.2f}")

    # tampilkan pembulatan jika ada
    if kolom_bulat:
        print(f"Pembulatan : {row[kolom_bulat]}")
    else:
        print("Pembulatan : (kolom tidak ditemukan)")

    print("\nKebutuhan Bahan :")
    for col in df_forecast.columns:
        if col.startswith("Kebutuhan_"):
            print(f"  - {col.replace('Kebutuhan_', ''):<15}: {row[col]}")

    print("\nSisa Stok :")
    for col in df_forecast.columns:
        if col.startswith("Sisa_"):
            print(f"  - {col.replace('Sisa_', ''):<15}: {row[col]}")

    print("-" * 60)

print("\n=== TANGGAL STOK MINIMUM TERCAPAI ===")
for bahan, tgl in batas_tanggal.items():
    print(f"{bahan:<15}: {tgl}")

# TAMBAHAN: CEK WAKTUNYA ORDER
order_flag = False

for col in df_forecast.columns:
    if col.startswith("Sisa_"):
        if row[col] <= 0:
            order_flag = True

if order_flag:
    print("⚠⚠ WAKTUNYA ORDER! Salah satu bahan mencapai batas minimum ⚠⚠")
else:
    print("Masih Aman ✔")

print("-" * 60)

=== HASIL PREDIKSI DENGAN BAHAN & STOK ===

Tanggal : 2025-11-28
Prediksi : 3.35
Pembulatan : 4.0

Kebutuhan Bahan :
  - Tepung         : 4.0
  - MargarinA      : 400.0
  - MargarinB      : 400.0
  - Telur          : 32.0
  - Air            : 3200.0
  - Gula           : 1600.0
  - Fermipan       : 4.0

Sisa Stok :
------------------------------------------------------------
Tanggal : 2025-11-29
Prediksi : 3.34
Pembulatan : 4.0

Kebutuhan Bahan :
  - Tepung         : 4.0
  - MargarinA      : 400.0
  - MargarinB      : 400.0
  - Telur          : 32.0
  - Air            : 3200.0
  - Gula           : 1600.0
  - Fermipan       : 4.0

Sisa Stok :
------------------------------------------------------------
Tanggal : 2025-11-30
Prediksi : 3.38
Pembulatan : 4.0

Kebutuhan Bahan :
  - Tepung         : 4.0
  - MargarinA      : 400.0
  - MargarinB      : 400.0
  - Telur          : 32.0
  - Air            : 3200.0
  - Gula           : 1600.0
  - Fermipan       : 4.0

Sisa Stok :
------------------